# Newton's Law of Cooling Simulation - Linear RNN Case

This notebook is intended to demonstrate manually setting the weights of an RNN with **linear activation** to reproduce a solution to Newton's Law of Cooling, then how to time warp to change the characteristic time scale. 

Continuous Solution:

$$
T(t) = e^{-kt}\;T_0 + (1 - e^{-kt})\;T_a
$$

Discretization Time Step with $\Delta t=1$, exact solution:

$$
T_{t+1} = e^{-k}\;T_{t} + (1 - e^{-k})\;T_a
$$

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as colors
import tensorflow as tf
import sys
sys.path.append("../src")
import reproducibility as reproducibility

In [ ]:
def newton_sol(T, Ta, k, t):
    """
    Discretization of exact solution with T(0)=T0
    """
    return np.exp(-k*t)*T + (1-np.exp(-k*t))*Ta

def evolve_temp(T0, Ta, k, nsteps):
    """
    Advance the temperature forward in time using fixed time steps. 
    This function is for use with changing Ta. 
    If Ta is fixed, this will return the same as if you directly evaluated
    newton_sol for each time.
    """
    T = np.zeros(nsteps)
    T[0] = T0
    
    for t in range(1, nsteps):
        T[t] = newton_sol(T[t-1], Ta[t], k, 1)

    return T

## Simulate and Visualize Solutions

In [ ]:
nsteps=50
k1=0.2
k2=.25*2
k3=.2/2
T0 = 50
Ta = np.repeat(20, nsteps)

In [ ]:
# Confirm two different methods of calculating deterministic curves
Ts = evolve_temp(T0, Ta, k1, nsteps)
Ts2 = np.array([newton_sol(T0, Ta[0], k1, t) for t in range(0, nsteps)])
print(np.max(np.abs(Ts - Ts2))) # Should be machine-epsilon

In [ ]:
# Plot grid of constants for demonstration
kgrid = np.linspace(start=0.5, stop=0.05, num=10)
Tk = np.zeros((len(kgrid), nsteps))

fig, ax = plt.subplots(figsize=(10, 6))

# colormap and normalization
cmap = cm.viridis
norm = colors.Normalize(vmin=kgrid.min(), vmax=kgrid.max())

for i, k in enumerate(kgrid):
    Tk[i] = evolve_temp(T0, Ta, k, nsteps)
    ax.plot(Tk[i],  color=cmap(norm(k)))
del k

# colorbar
sm = cm.ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])  # required for older matplotlib
cbar = fig.colorbar(sm, ax=ax)
cbar.set_label("Cooling Constant k")

ax.set_xlabel("Time step")
ax.set_ylabel("Temperature")
plt.title("Simulated from Newton's Law")
plt.show()

In [ ]:
print(len(kgrid))
print(kgrid[0])
print(kgrid[-1])

## Simple RNN Case

Fixed K, 1 recurrent cell

### MSE Loss can't Learn a Smooth Exponential

Networks will tend to learn to output a constant sequence corresponding to the overall signal mean. 

In [ ]:
reproducibility.set_seed(123)

inputs = tf.keras.Input(batch_shape=(None, nsteps, 1))
x = tf.keras.layers.SimpleRNN(1, return_sequences=True, activation="tanh")(inputs)
# x = tf.keras.layers.LSTM(1, return_sequences=True, activation="tanh")(inputs)
outputs = tf.keras.layers.Dense(1)(x)
rnn = tf.keras.Model(inputs, outputs)
opt = tf.keras.optimizers.Adam(learning_rate=0.1)
rnn.compile(loss = "mean_squared_error", optimizer=opt)
rnn.summary()

In [ ]:
y1 = np.array([newton_sol(T0, Ta[0], k1, t) for t in range(0, nsteps)])

N = 100  # hyperparameter: number of duplicated sequences
Xrep = np.full((N, nsteps, 1), Ta[0], dtype=np.float32)
Yrep = np.repeat(y1[None, :, None], N, axis=0)

In [ ]:
rnn.fit(Xrep, Xrep, epochs=N, verbose=False)

In [ ]:
preds = rnn.predict(Xrep[0,:,:])

In [ ]:
plt.plot(preds.flatten(), label="RNN Fit")
plt.plot(y1, label="Simulated")
plt.grid()
plt.legend()

### Manual Initialization

Setting weights of RNN without training to exactly solve the system. Just using as a method of empirical validation of scaling factors.

Discrete solution in terms of weight matrices:

\begin{aligned}
    T_{t+1} &= e^{-k}\;T_{t} + (1 - e^{-k})\;T_a\\
    &=\alpha\;T_{t} + (1 - \alpha)\;T_a, \qquad \alpha=e^{-k}\\
    &= W_h T_t + W_x T_a
\end{aligned}

Initial Hidden state: we wish to fix $T_0$, so time shift the simulation right by 1

In [ ]:
reproducibility.set_seed(123)

B = 1 # batch size
h0 = tf.fill((B, 1), T0) # Initial hidden state
inputs = tf.keras.Input(batch_shape=(None, nsteps-1, 1))
rnn_layer = tf.keras.layers.SimpleRNN(
    1,
    return_sequences=True,
    activation="linear"
)
x = rnn_layer(inputs, initial_state=h0)

outputs = tf.keras.layers.Dense(1)(x)
rnn = tf.keras.Model(inputs, outputs)
rnn.compile(loss = "mean_squared_error", optimizer="Adam")
rnn.summary()

In [ ]:
# Set simple RNN weights 
alpha=np.exp(-k1)

rweights = rnn.get_weights()
rweights[0] = np.array([[(1-alpha)]]) # Input
rweights[1] = np.array([[alpha]]) # Recurrent connection
rweights[2] = np.array([0])    # RNN Cell Bias
rweights[3] = np.array([[1]])  # Dense Output Activation
rweights[4] = np.array([0])  # Dense Output Bias

rnn.set_weights(rweights)

In [ ]:
X = np.full((1, nsteps-1, 1), Ta[0], dtype=np.float32)
pred1 = rnn.predict(X, verbose=0)
pred1 = np.concatenate(([T0], pred1.flatten()))

In [ ]:
plt.plot(pred1, '-', label="RNN Fit", linewidth=3)
plt.plot(y1, '--', label="Simulated", linewidth=3)
plt.grid()
plt.legend()

### Time Warp

Modify $W_x$ and $W_h$, leave all others unchanged. The weights can be set with exact knowledge of $k$, but as a demonstration we set them in terms of a parameter $\gamma$, which is defined to be the time-warp factor. 
- For iteration 0, set weights with $e^{-k}$
- For iterations 1 through all, let $\tilde k$ be time-warped constant
    - calculate $\gamma = \tilde k/k$
    - Modify weights $\tilde W_h = (W_h)^{\gamma}$ and $\tilde W_x = 1-(1-W_x)^\gamma$

Error between true curves and RNN predictions is driven by computational rounding effects. These computational errors include the evaluation of the exponential calculation $e^{-k}$. Then, the iterative process of generating the RNN predictions compounds these computational errors.

In [ ]:
kcurves = {} # object for storing RNN preds

k0 = kgrid[0]
alpha=np.exp(-k0)
rweights0  = [w.copy() for w in rnn.get_weights()]
rweights0[0] = np.array([[(1-alpha)]]) # Input weight
rweights0[1] = np.array([[alpha]]) # Recurrent connection 
rnn.set_weights(rweights0)
predk = rnn.predict(X, verbose=0)
predk = np.concatenate(([T0], predk.flatten()))

kcurves[f"k0"] = predk

for k in kgrid[1:]:
    g = k / k0
    # alpha2=np.exp(-k) # NOT used, just here to show you could use alpha directly to get same result
    rweights = rnn.get_weights()
    rweights[0] = 1-(1-rweights0[0])**g
    rweights[1] = rweights0[1]**g   
    rnn.set_weights(rweights)
    predk = rnn.predict(X, verbose=0)
    predk = np.concatenate(([T0], predk.flatten()))
    
    kcurves[f"k.{k}"] = predk

kcurves = np.stack(list(kcurves.values()), axis=0)

In [ ]:
print(f"Max absolute difference to theoretical curves: {np.max(np.abs(kcurves - Tk))}")

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import cm, colors

# document-safe defaults (local, not global rcParams)
FIGSIZE = (10, 6)
DPI = 300
LABEL_SIZE = 14
TICK_SIZE = 12
CBAR_LABEL_SIZE = 13

fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)

# colormap and normalization
cmap = cm.viridis
norm = colors.Normalize(vmin=kgrid.min(), vmax=kgrid.max())

for i, k in enumerate(kgrid):
    ax.plot(kcurves[i], color=cmap(norm(k)))

# colorbar
sm = cm.ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax)
cbar.set_label("Cooling Constant k", fontsize=CBAR_LABEL_SIZE)
cbar.ax.tick_params(labelsize=TICK_SIZE)

# axes labels + ticks
ax.set_xlabel("Time step", fontsize=LABEL_SIZE)
ax.set_ylabel("Temperature", fontsize=LABEL_SIZE)
ax.tick_params(axis="both", labelsize=TICK_SIZE)

fig.tight_layout()
fig.savefig("outputs/rnn_kcurves.png", dpi=DPI, bbox_inches="tight")
plt.show()
plt.close(fig)

## LSTM Case

Fix k, 1 recurrent cell, 1 dense cell. 

First method: linear activation, can just manuallys set

Second method: tanh activation
- Set up weights to approximate $tanh()$ of the system
- Freeze recurrent layer, allow dense layer to learn mapping from (-1,1) to the true values.

Error between true curves and LSTM predictions is partially driven by the sigmoid gates approximating 1. This is in addition to the computational errors mentioned above for the simple RNN.

In [ ]:
reproducibility.set_seed(123)

B = 1 # batch size
h0 = tf.zeros((B, 1))          # initial hidden state
c0 = tf.fill((B, 1), T0)       # initial cell state

inputs = tf.keras.Input(batch_shape=(None, nsteps-1, 1))
lstm_layer = tf.keras.layers.LSTM(
    1,
    return_sequences=True,
    activation="linear"
)
x = lstm_layer(inputs, initial_state=[h0, c0])

outputs = tf.keras.layers.Dense(1)(x)
lstm = tf.keras.Model(inputs, outputs)
lstm.compile(loss = "mean_squared_error", optimizer="Adam")
lstm.summary()

In [ ]:
alpha=np.exp(-k1)

lweights = lstm.get_weights()
# Set Input Weights, 1 feature, 4 inputs
lweights[0] = np.array([[ 
    0.0,   # input gate
    0.0,   # forget gate
    1.0,   # candidate gate
    0.0    # output gate
]])

# Set Recurrent Weights
lweights[1] = np.array([[
    0,                 # Input gate
    0,                 # Forget gate
    0,                 # Candidate
    0,                 # Output gate
]])

bf = np.log(alpha / (1 - alpha))
bi = np.log((1 - alpha) / alpha)

# Set Biases
lweights[2] = np.array([
    bi,     # input gate bias, σ(bi) = 1 - alpha
    bf,     # forget gate bias, σ(bf) = alpha
    0.0,    # candidate bias
    10.0    # output gate bias, σ approx 1
])

# Set output neuron weights(1x1)
lweights[3] = np.array([[1.0]])

# Dense bias (length-1 vector)
lweights[4] = np.array([0.0])

lstm.set_weights(lweights)

In [ ]:
pred1 = lstm.predict(X, verbose=0)
pred1 = np.concatenate(([T0], pred1.flatten()))

In [ ]:
plt.plot(pred1, '-', label="LSTM Fit", linewidth=3)
plt.plot(y1, '--', label="Simulated", linewidth=3)
plt.grid()
plt.legend()

### Time warp

In [ ]:
# Helper logit and sigmoid functions to make code clearer

def logit(x):
    return np.log(x / (1 - x))

def sig(x):
    return 1 / (1 + np.exp(-x))

In [ ]:
print(f"Output gate activation: {sig(10)}")

In [ ]:
kcurves = {} # object for storing RNN preds

k0 = kgrid[0]
alpha=np.exp(-k0)
lweights0  = [w.copy() for w in lstm.get_weights()]
bi = logit(1-alpha)
bf = logit(alpha)
# Set Biases
lweights0[2] = np.array([
    bi,     # input gate bias, σ(bi) = 1 - alpha
    bf,     # forget gate bias, σ(bf) = alpha
    0.0,    # candidate bias
    10.0    # output gate bias, σ approx 1
])

lstm.set_weights(lweights0)
predk = lstm.predict(X, verbose=0)
predk = np.concatenate(([T0], predk.flatten()))

kcurves[f"k0"] = predk

for k in kgrid[1:]:
    g = k / k0
    
    
    lweights = lstm.get_weights()
    ## Commented block below is how to directly calculate from knowledge of k
    # alpha2=np.exp(-k)
    # bf = np.log(alpha2 / (1 - alpha2))
    # bi = np.log((1 - alpha2) / alpha2)

    bi_0 = lweights0[2][0]
    bf_0 = lweights0[2][1]

    bi_new = logit(1-(1-sig(bi_0))**g)
    bf_new = logit(sig(bf_0)**g)
    lweights[2] = np.array([
        bi_new,     # input gate bias, σ(bi) = 1 - alpha
        bf_new,     # forget gate bias, σ(bf) = alpha
        0.0,    # candidate bias
        10.0    # output gate bias, σ approx 1
    ]) 
    lstm.set_weights(lweights)
    predk = lstm.predict(X, verbose=0)
    predk = np.concatenate(([T0], predk.flatten()))
    
    kcurves[f"k.{k}"] = predk

kcurves = np.stack(list(kcurves.values()), axis=0)

In [ ]:
print(f"Max absolute difference to theoretical curves: {np.max(np.abs(kcurves - Tk))}")

In [ ]:
print(logit(1-0.003/50))

$b_o > \text{logit}(1-\epsilon/M)$

In [ ]:
# By construction
print("epsilon = 0.003")

$10 > logit(1-0.003/50) \approx 9.7$

$\sigma(b_o) > 1-\epsilon/M$

$\epsilon/M > 1-\sigma(b_o)$

$\epsilon > M(1-\sigma(b_o))$

In [ ]:
M = np.max(kcurves)
print(M)

In [ ]:
fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)

# colormap and normalization
cmap = cm.viridis
norm = colors.Normalize(vmin=kgrid.min(), vmax=kgrid.max())

for i, k in enumerate(kgrid):
    ax.plot(kcurves[i], color=cmap(norm(k)))

# colorbar
sm = cm.ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax)
cbar.set_label("Cooling Constant k", fontsize=CBAR_LABEL_SIZE)
cbar.ax.tick_params(labelsize=TICK_SIZE)

# axes labels + ticks
ax.set_xlabel("Time step", fontsize=LABEL_SIZE)
ax.set_ylabel("Temperature", fontsize=LABEL_SIZE)
ax.tick_params(axis="both", labelsize=TICK_SIZE)